[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/12_linear_attention_solution.ipynb)

# ✅ Solution: linear_attention

Implement **Linear Attention** — O(S·D²) instead of O(S²·D), enabling efficient long-sequence processing.

Replace softmax with a **kernel feature map** $\phi$:

$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

### Feature map
Use $\phi(x) = \text{elu}(x) + 1$ (ensures non-negative features).

### Signature
```python
def linear_attention(q_BLK, k_BLK, v_BLK):
    # Q: (B, S, D_k), K: (B, S, D_k), V: (B, S, D_v)
    # Returns: (B, S, D_v)
```

### Key insight
Instead of computing the $S \times S$ attention matrix, compute $\phi(K)^T V$ first (a $D_k \times D_v$ matrix), then multiply by $\phi(Q)$.

### Rules
- Must use a feature map (NOT softmax)
- Must be O(S·D²) — should run fast on long sequences
- You **may** use `F.elu`


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import math


In [ ]:
# ✅ SOLUTION

import jax, jax.numpy as jnp
def linear_attention(q_BLK, k_BLK, v_BLK):
    phi_q_BLK = jax.nn.elu(q_BLK) + 1
    phi_k_BLK = jax.nn.elu(k_BLK) + 1
    kv_BKD = jnp.swapaxes(phi_k_BLK, -2, -1) @ v_BLK
    z_B1K = jnp.sum(phi_k_BLK, axis=1, keepdims=True)
    return (phi_q_BLK @ kv_BKD) / (phi_q_BLK @ jnp.swapaxes(z_B1K, -2, -1) + 1e-6)


In [ ]:
# Verify
print(linear_attention)


In [ ]:
from jax_judge import check
check("linear_attention")
